# BSF Larvae Image Processing - Notebook Version

This notebook processes Black Soldier Fly larvae images with **GPU/CPU performance tracking**.

**What it does:**
1. Connect to your Google Drive (on Colab) or use local files (on PC)
2. Pick your model and image files from Drive/local storage
3. Detect larvae using YOLO
4. Extract tray number using OCR
5. Calculate measurements (length, width, area, weight) for each larva
6. **Calculate and upload AVERAGE measurements to dashboard**
7. **Track processing time to compare GPU vs CPU**

**Data uploaded:** Average length, width, area, weight + total count per tray

**On Colab:** Upload files to Google Drive first - no need to re-upload every time!
**On Local PC:** Just provide file paths when prompted

---

## 📂 Google Drive Setup - Pick Files from Your Drive!

**On Google Colab:**
1. Enable GPU first: `Runtime` → `Change runtime type` → Select `GPU` (T4 GPU is free)
2. Upload your files to Google Drive first:
   - `flat_bug.pt` (23MB YOLO model)
   - Your larvae image (JPG/PNG)
3. Run the cell below - it will mount your Google Drive
4. Enter the paths to your files (e.g., `/content/drive/MyDrive/flat_bug.pt`)
5. **No need to re-upload files every time - they're saved in your Drive!**

**On Local PC:**
- Just enter the file paths when prompted

**Speed comparison (Pi5 vs Colab GPU):**
- Raspberry Pi 5 (CPU): ~5-10 seconds per image
- Google Colab (T4 GPU): ~0.5-1 second per image ⚡ **10-20x faster!**

---

In [ ]:
# File Setup - Works on both Colab (Google Drive) and Local PC
import time

# Detect if running on Colab
try:
    from google.colab import drive
    RUNNING_ON_COLAB = True
    print("🌐 Running on Google Colab")
    
    # Mount Google Drive
    print("\n📂 Mounting Google Drive...")
    drive.mount('/content/drive')
    print("✅ Google Drive mounted at /content/drive")
    print("   Your files are in: /content/drive/MyDrive/")
    
except ImportError:
    RUNNING_ON_COLAB = False
    print("💻 Running on Local PC")

# Get YOLO Model Path
print("\n📁 Step 1: YOLO Model (flat_bug.pt - 23MB)")
if RUNNING_ON_COLAB:
    print("   Example: /content/drive/MyDrive/flat_bug.pt")
    model_path = input("   Enter path in your Google Drive: ")
else:
    print("   Example: /home/user/Desktop/flat_bug.pt")
    model_path = input("   Enter local path: ")
print(f"✅ Model path: {model_path}")

# Get Larvae Image Path
print("\n📸 Step 2: Larvae Image")
if RUNNING_ON_COLAB:
    print("   Example: /content/drive/MyDrive/larvae_image.jpg")
    image_path = input("   Enter path in your Google Drive: ")
else:
    print("   Example: /home/user/Pictures/larvae.jpg")
    image_path = input("   Enter local path: ")
print(f"✅ Image path: {image_path}")

# Check GPU availability
try:
    import torch
    if torch.cuda.is_available():
        print(f"\n🚀 GPU detected: {torch.cuda.get_device_name(0)}")
        print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    else:
        print("\n⚠️  No GPU detected - using CPU (slower)")
except ImportError:
    print("\n⚠️  PyTorch not imported yet - GPU check will happen after install")

## Step 1: Install Required Packages
Run this cell once to install all necessary libraries.

In [ ]:
!pip install ultralytics opencv-python easyocr Pillow numpy requests matplotlib

## Step 2: Import Libraries
Import all necessary libraries for image processing.

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import easyocr
from PIL import Image
import matplotlib.pyplot as plt
import requests
import json
from datetime import datetime
import os
import time  # For tracking processing time

## Step 3: Load YOLO Model
Load the flat_bug model for larvae detection. Make sure `flat_bug.pt` is in the same directory as this notebook.

In [ ]:
# Start timing
start_time = time.time()

# Load YOLO model for larvae detection
print(f"Loading YOLO model from: {model_path}")
model = YOLO(model_path)
model_load_time = time.time() - start_time
print(f"✅ YOLO model loaded in {model_load_time:.2f}s")

# Initialize EasyOCR reader for tray number detection
ocr_start = time.time()
reader = easyocr.Reader(['en'], gpu=True)  # Set gpu=False if no GPU available
ocr_load_time = time.time() - ocr_start
print(f"✅ EasyOCR reader initialized in {ocr_load_time:.2f}s")

print(f"\n⏱️  Total setup time: {time.time() - start_time:.2f}s")

## Step 4: Load Your Image

**If you uploaded in Colab:** The image_path is already set - just run this cell to load it.

**If running locally:** Enter the full path to your image file.

In [ ]:
# Load the image (path was already set in upload cell)
if os.path.exists(image_path):
    image = cv2.imread(image_path)
    print(f"✅ Image loaded: {image_path}")
    print(f"   Size: {image.shape[1]}x{image.shape[0]} pixels")
    
    # Display the original image
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title("Original Image")
    plt.axis('off')
    plt.show()
else:
    print(f"❌ Error: File not found at {image_path}")
    print("   Please check the path and try again.")

## Step 5: Detect Tray Number with OCR
Extract the tray number from the image using EasyOCR.

In [ ]:
# Extract tray number using OCR
ocr_start_time = time.time()
ocr_result = reader.readtext(image)
ocr_time = time.time() - ocr_start_time

# Find numbers in OCR results
tray_number = None
for (bbox, text, prob) in ocr_result:
    # Look for 5-digit numbers (tray IDs are typically 5 digits)
    text_clean = ''.join(filter(str.isdigit, text))
    if len(text_clean) == 5 and prob > 0.5:
        tray_number = int(text_clean)
        print(f"✅ Tray number detected: {tray_number} (confidence: {prob:.2f})")
        break

if tray_number is None:
    print("⚠️ No tray number detected, using default: 0")
    tray_number = 0

print(f"⏱️  OCR processing time: {ocr_time:.2f}s")

## Step 6: Detect Larvae with YOLO
Run YOLO detection to find all larvae in the image.

In [ ]:
# Run YOLO detection
yolo_start_time = time.time()
results = model(image, conf=0.25)
yolo_time = time.time() - yolo_start_time
print(f"⏱️  YOLO detection time: {yolo_time:.2f}s")

# Process detections
larvae_data = []
for result in results:
    boxes = result.boxes
    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = box.conf[0].cpu().numpy()
        
        # Calculate measurements
        length_px = abs(y2 - y1)
        width_px = abs(x2 - x1)
        area_px = length_px * width_px
        
        # Convert pixels to mm (assuming 1px = 0.1mm, adjust based on your camera)
        PX_TO_MM = 0.1
        length_mm = length_px * PX_TO_MM
        width_mm = width_px * PX_TO_MM
        area_mm2 = area_px * (PX_TO_MM ** 2)
        
        # Estimate weight using empirical formula
        # weight (mg) ≈ 0.001 * area (mm²)
        weight_mg = 0.001 * area_mm2
        
        larvae_data.append({
            'bbox': [int(x1), int(y1), int(x2), int(y2)],
            'confidence': float(conf),
            'length': float(length_mm),
            'width': float(width_mm),
            'area': float(area_mm2),
            'weight': float(weight_mg)
        })

print(f"✅ Detected {len(larvae_data)} larvae!")

## Step 7: Visualize Detected Larvae

In [ ]:
# Draw bounding boxes on image
annotated_image = image.copy()
for larva in larvae_data:
    x1, y1, x2, y2 = larva['bbox']
    cv2.rectangle(annotated_image, (x1, y1), (x2, y2), (0, 255, 0), 2)
    # Add label with weight
    label = f"{larva['weight']:.2f}mg"
    cv2.putText(annotated_image, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

# Display annotated image
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB))
plt.title(f"Tray {tray_number} - Detected {len(larvae_data)} Larvae")
plt.axis('off')
plt.show()

# Show statistics
if larvae_data:
    weights = [larva['weight'] for larva in larvae_data]
    lengths = [larva['length'] for larva in larvae_data]
    print(f"\n📊 Statistics:")
    print(f"   Total larvae: {len(larvae_data)}")
    print(f"   Average weight: {np.mean(weights):.2f} mg")
    print(f"   Average length: {np.mean(lengths):.2f} mm")
    print(f"   Total weight: {np.sum(weights):.2f} mg")

# Show total processing time
total_time = yolo_time + ocr_time
print(f"\n⏱️  TOTAL PROCESSING TIME: {total_time:.2f}s")
print(f"   - OCR: {ocr_time:.2f}s")
print(f"   - YOLO: {yolo_time:.2f}s")

## Step 8: Upload Results to Dashboard

Upload the average measurements to your BSF dashboard at http://134.122.83.13:8000

In [ ]:
# Prepare average data for upload
api_url = "http://134.122.83.13:8000/api/larvae_data"

if larvae_data:
    # Calculate averages
    avg_length = np.mean([larva["length"] for larva in larvae_data])
    avg_width = np.mean([larva["width"] for larva in larvae_data])
    avg_area = np.mean([larva["area"] for larva in larvae_data])
    avg_weight = np.mean([larva["weight"] for larva in larvae_data])
    total_count = len(larvae_data)
    
    # Format data for API (sending averages)
    upload_data = {
        "tray_number": tray_number,
        "length": round(avg_length, 2),
        "width": round(avg_width, 2),
        "area": round(avg_area, 2),
        "weight": round(avg_weight, 2),
        "count": total_count,
        "timestamp": datetime.now().isoformat()
    }
    
    # Upload to API
    print(f"\n⬆️  Uploading average data for {total_count} larvae to dashboard...")
    print(f"   Avg Length: {avg_length:.2f} mm")
    print(f"   Avg Width: {avg_width:.2f} mm")
    print(f"   Avg Area: {avg_area:.2f} mm²")
    print(f"   Avg Weight: {avg_weight:.2f} mg")
    
    try:
        response = requests.post(api_url, json=upload_data)
        
        if response.status_code == 201:
            print(f"\n✅ Successfully uploaded data for Tray {tray_number}!")
            print(f"   View at: http://134.122.83.13:8000/dashboard")
        else:
            print(f"\n❌ Upload failed: {response.status_code}")
            print(f"   Response: {response.text}")
    except Exception as e:
        print(f"\n❌ Error uploading: {str(e)}")
else:
    print("\n⚠️  No larvae detected - skipping upload")

---

## 💡 Tips & Notes

### How to Use on Google Colab
1. **Upload files to your Google Drive first:**
   - Upload `flat_bug.pt` (23MB) to your Google Drive
   - Upload your larvae images to Google Drive
   
2. **Run the notebook:**
   - Upload this notebook to Colab
   - Enable GPU: `Runtime` → `Change runtime type` → `GPU`
   - Run the first cell - it will mount your Google Drive
   - Enter the paths (e.g., `/content/drive/MyDrive/flat_bug.pt`)
   
3. **Benefits:**
   - Files stay in your Drive - no need to re-upload every time!
   - Process multiple images by re-running from Step 4
   - Check timing results to see GPU speed improvement

### Processing Time Comparison
The notebook tracks exact processing time:
- **OCR time** - tray number detection
- **YOLO time** - larvae detection
- **Total time** - complete processing

**Expected results:**
- Pi5 CPU: ~5-10 seconds total
- Colab GPU: ~0.5-1 second total ⚡

### Adjusting the Pixel-to-MM Conversion
The default `PX_TO_MM = 0.1` may need adjustment based on your camera setup. To calibrate:
1. Place an object of known size in the image
2. Measure its pixel dimensions
3. Calculate: `PX_TO_MM = actual_mm / pixel_count`

### Processing Multiple Images
To process multiple images on Colab:
- Just re-run from Step 4 onwards
- Enter a different image path from your Drive
- All results are timed individually

### What Gets Uploaded to Dashboard
The notebook sends **average measurements** for each tray:
- **Average length** (mm) - mean of all detected larvae
- **Average width** (mm) - mean of all detected larvae
- **Average area** (mm²) - mean of all detected larvae
- **Average weight** (mg) - mean of all detected larvae
- **Count** - total number of larvae detected
- **Tray number** - from OCR detection
- **Timestamp** - when the processing occurred

Individual larvae measurements are shown locally but only averages are uploaded.

### Troubleshooting
- **No larvae detected**: Lower the confidence threshold (change `conf=0.25` to `conf=0.15`)
- **OCR not detecting tray number**: Check that the tray number is clearly visible and well-lit
- **Upload fails**: Make sure you're logged in to the dashboard first
- **File not found**: Make sure the path starts with `/content/drive/MyDrive/`

### Files Needed
**For Colab:** Upload to Google Drive first
- `flat_bug.pt` - YOLO model file (23MB)
- Your BSF larvae images (JPG/PNG)

**For Local PC:**
- Same files, stored on your local drive